In [16]:
import numpy as np
from scipy.interpolate import griddata
import numpy as np
import torch
import sys, os
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
from datasets.data import SpatialDataset
from datasets.transformerRegressorDataClass import TransformerPointDataset
from scipy.interpolate import Rbf
from scipy.interpolate import LinearNDInterpolator

/home/user_116/Project-B-Technion


In [ ]:
def _to_numpy(x):
    # works for numpy, torch tensors, lists
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return np.asarray(x)

def _scalar(x):
    if x is None:
        return np.nan
    a = np.asarray(x)
    if a.size == 0:
        return np.nan
    # robustly take the first element (handles (), (1,), (1,1), etc.)
    return float(a.ravel()[0])


def idw_local_all(nei_coords_list, nei_y_list, query_coords, p=2.0, eps=1e-12):
    """
    Local IDW using *all* precomputed neighbors for each target point.
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)

    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)
        #print(nei_xy.size)
        if nei_xy.size == 0:
            preds[i] = np.nan
            continue

        d = np.linalg.norm(nei_xy - qxy[i], axis=1)  # (S,)

        # exact match protection
        if np.any(d < 1e-12):
            preds[i] = float(nei_y[d.argmin()])
            continue

        w = 1.0 / (d**p + eps)
        preds[i] = float(np.sum(w * nei_y) / np.sum(w))

    return preds

def linear_local(nei_coords_list, nei_y_list, query_coords, fill_with_idw=True, p=2.0):
    """
    Local linear interpolation using precomputed neighbors.
    - nei_coords_list: list of (S_i, 2)
    - nei_y_list: list of (S_i,)
    - query_coords: (N, 2)
    - fill_with_idw: if True, fill NaN/extrapolation with local IDW
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)

    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)

        if len(nei_xy) < 3:
            # not enough points to form a triangle; fallback to mean
            preds[i] = np.mean(nei_y)
            continue

        try:
            interp = LinearNDInterpolator(nei_xy, nei_y, fill_value=np.nan)
            pred = interp(qxy[i])
            if np.isnan(pred) and fill_with_idw:
                # fallback to local IDW if outside convex hull
                d = np.linalg.norm(nei_xy - qxy[i], axis=1)
                w = 1.0 / (d**p + 1e-12)
                pred = np.sum(w * nei_y) / np.sum(w)
        except Exception:
            # triangulation sometimes fails if neighbors are colinear
            d = np.linalg.norm(nei_xy - qxy[i], axis=1)
            w = 1.0 / (d**p + 1e-12)
            pred = np.sum(w * nei_y) / np.sum(w)

        preds[i] = _scalar(pred)
    return preds

def rbf_local(nei_coords_list, nei_y_list, query_coords, function='linear'):
    preds = np.zeros(len(nei_coords_list))
    qxy = _to_numpy(query_coords)
    for i, (xy, y) in enumerate(zip(nei_coords_list, nei_y_list)):
        xy = _to_numpy(xy); y = _to_numpy(y).reshape(-1)
        if len(xy) < 3:
            preds[i] = np.mean(y)
            continue
        try:
            rbf = Rbf(xy[:,0], xy[:,1], y, function=function)
            preds[i] = float(rbf(qxy[i,0], qxy[i,1]))
        except Exception:
            preds[i] = np.mean(y)
    return preds

def MSE(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def MAP(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred))/len(y_true)

In [6]:
# Show the current working directory
from pathlib import Path
print(os.getcwd())
os.chdir("..")
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
p_cache = Path("cache")
print("cache exists?", p_cache.exists(), "->", p_cache.resolve())
trainset = torch.load(r"./cache/trainset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.001_seed5.pt",map_location="cpu",weights_only=False)
validset = torch.load(r"./cache/validset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.001_seed5.pt",map_location="cpu",weights_only=False)
testset = torch.load(r"./cache/testset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.001_seed5.pt",map_location="cpu",weights_only=False)

/home/user_116/Project-B-Technion/Transformer_Map_Interp/models
/home/user_116/Project-B-Technion
cache exists? True -> /home/user_116/Project-B-Technion/Transformer_Map_Interp/cache


In [36]:
print("Performing Local IDW interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = idw_local_all(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = idw_local_all(testset.obs_coords_norm, testset.obs_y_norm,
                     test_query_coords)

Performing Local IDW interpolation on validation and test sets...
208
218
186
266
246
238
228
86
228
258
198
224
204
204
244
192
248
218
218
228
262
220
240
204
228
240
200
246
212
222
234
196
224
246
182
238
212
260
210
126
220
258
194
188
210
206
118
210
220
230
100
204
190
232
232
202
208
212
220
216
238
258
144
208
188
250
196
258
190
200
212
254
220
212
182
204
250
194
206
238
224
100
216
218
198
208
208
228
200
214
198
236
206
252
208
190
214
242
226
234
210
222
226
234
208
220
196
208
230
216
240
198
218
204
170
206
194
218
220
216
204
244
206
240
226
242
238
236
224
218
266
230
186
226
206
212
138
166
220
244
206
216
214
228
232
226
238
244
210
216
198
216
236
108
228
248
196
252
224
228
196
260
226
186
256
254
186
272
228
218
204
226
210
220
232
196
214
172
228
196
232
206
220
212
244
218
186
218
214
202
236
250
242
230
174
176
210
214
200
236
220
200
234
248
218
218
270
190
182
252
222
230
224
220
198
224
250
214
226
208
226
202
228
220
250
224
186
208
98
220
206
190
220
234


In [42]:
print("Performing Local Linear interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = linear_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = linear_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local Linear interpolation on validation and test sets...


In [39]:
print("Performing Local RBF interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = rbf_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = rbf_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local RBF interpolation on validation and test sets...


In [37]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local IDW] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local IDW] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[Local IDW] Validation MSE=0.003, MAE=0.040
[Local IDW] Test MSE=0.002, MAE=0.024


In [38]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
mse_val = np.mean((val_pred_unnorm - validset.query_y.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred_unnorm - validset.query_y.detach().cpu().numpy()))
mse_test = np.mean((test_pred_unnorm - testset.query_y.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred_unnorm - testset.query_y.detach().cpu().numpy()))
print(f"[Local IDW] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local IDW] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[Local IDW] Validation MSE=351895.165, MAE=525.818
[Local IDW] Test MSE=191822.336, MAE=346.622


In [43]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local Linear] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local Linear] Test MSE={mse_test:.4f}, MAE={mae_test:.4f}")

[Local Linear] Validation MSE=0.001, MAE=0.018
[Local Linear] Test MSE=0.0004, MAE=0.0105


In [44]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
mse_val = np.mean((val_pred_unnorm - validset.query_y.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred_unnorm - validset.query_y.detach().cpu().numpy()))
mse_test = np.mean((test_pred_unnorm - testset.query_y.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred_unnorm - testset.query_y.detach().cpu().numpy()))
print(f"[Local Linear] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local Linear] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[Local Linear] Validation MSE=352731.390, MAE=526.213
[Local Linear] Test MSE=192377.526, MAE=346.826


In [40]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.4f}, MAE={mae_test:.4f}")

[Local RBF] Validation MSE=0.001, MAE=0.016
[Local RBF] Test MSE=0.0003, MAE=0.0094


In [41]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
mse_val = np.mean((val_pred_unnorm - validset.query_y.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred_unnorm - validset.query_y.detach().cpu().numpy()))
mse_test = np.mean((test_pred_unnorm - testset.query_y.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred_unnorm - testset.query_y.detach().cpu().numpy()))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[Local RBF] Validation MSE=352603.694, MAE=526.074
[Local RBF] Test MSE=192370.897, MAE=346.822
